<div style="background: linear-gradient(135deg, #003A70 0%, #004E8C 40%, #006847 100%);
border-radius: 12px; padding: 50px 40px 40px; margin: 0 0 30px;
color: white; text-align: center; box-shadow: 0 4px 20px rgba(0,0,0,0.15);">

<div style="font-size: 0.85em; letter-spacing: 4px; text-transform: uppercase;
color: rgba(255,255,255,0.7); margin-bottom: 10px;">Proyecto Integrador</div>

<h1 style="font-size: 3em; font-weight: 800; color: white; margin: 0 0 5px;
letter-spacing: 1px;">EpiForecast-MX</h1>

<div style="width: 80px; height: 3px; background: #9B2242; margin: 15px auto; border-radius: 2px;"></div>

<h2 style="font-size: 1.4em; font-weight: 400; color: rgba(255,255,255,0.95);
margin: 15px 0 25px;">Avance 4: Modelos Alternativos y Selecci&oacute;n del Modelo Final</h2>

<div style="display: inline-block; background: rgba(255,255,255,0.12);
border-radius: 8px; padding: 10px 28px; margin: 0 0 25px;">
<span style="font-size: 1.05em; color: rgba(255,255,255,0.95); letter-spacing: 1px;">
Depresi&oacute;n [F32] &middot; Parkinson [G20] &middot; Alzheimer [G30]</span>
</div>

<p style="font-size: 1.05em; color: rgba(255,255,255,0.85); margin: 0 0 5px;">
<strong>Maestr&iacute;a en Inteligencia Artificial Aplicada</strong></p>
<p style="font-size: 0.95em; color: rgba(255,255,255,0.7); margin: 0 0 30px;">
Tecnol&oacute;gico de Monterrey &mdash; TC5035</p>

<table style="margin: 0 auto 25px; border-collapse: collapse; font-size: 0.95em;
min-width: 340px;">
<tr style="border-bottom: 2px solid rgba(255,255,255,0.3);">
  <th style="padding: 10px 20px; text-align: left; color: rgba(255,255,255,0.8);
font-weight: 600; letter-spacing: 1px; text-transform: uppercase; font-size: 0.8em;">Integrante</th>
  <th style="padding: 10px 20px; text-align: left; color: rgba(255,255,255,0.8);
font-weight: 600; letter-spacing: 1px; text-transform: uppercase; font-size: 0.8em;">Afiliaci&oacute;n</th>
</tr>
<tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
<td style="padding: 8px 20px; color: white;">Javier Rebull</td>
<td style="padding: 8px 20px; color: rgba(255,255,255,0.8);">Tec / Santander US</td></tr>
<tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
<td style="padding: 8px 20px; color: white;">Juan Carlos P&eacute;rez Nava</td>
<td style="padding: 8px 20px; color: rgba(255,255,255,0.8);">Tec / IMSS</td></tr>
<tr>
<td style="padding: 8px 20px; color: white;">Luis S&aacute;nchez</td>
<td style="padding: 8px 20px; color: rgba(255,255,255,0.8);">Tec / Tesla</td></tr>
</table>

<p style="color: rgba(255,255,255,0.5); font-size: 0.85em; margin: 0;">
Equipo 01 &middot; Febrero 2026</p>

</div>

### Contenido

| # | Secci&oacute;n | Figuras |
|---|---------|---------|
| 1 | [Contexto del proyecto](#sec1) | &mdash; |
| 2 | [Recorrido de optimizaci&oacute;n: v1 a v6](#sec2) | Fig 1 |
| 3 | [Metodolog&iacute;a Prophet](#sec3) | Fig 2 |
| 4 | [Resultados de producci&oacute;n: 312 modelos](#sec4) | Fig 3-5 |
| 5 | [Galer&iacute;a de pron&oacute;sticos](#sec5) | Fig 6-9 |
| 6 | [Benchmark SageMaker: 6 algoritmos](#sec6) | Fig 10-13 |
| 7 | [An&aacute;lisis por padecimiento](#sec7) | Fig 14-15 |
| 8 | [An&aacute;lisis por sexo](#sec8) | Fig 16 |
| 9 | [Predicciones cara a cara](#sec9) | Fig 17-18 |
| 10 | [Selecci&oacute;n del modelo final](#sec10) | Fig 19-20 |
| 11 | [Dashboard Tableau](#sec11) | &mdash; |
| 12 | [Publicaci&oacute;n acad&eacute;mica](#sec12) | &mdash; |
| 13 | [Conclusiones](#sec13) | &mdash; |
| 14 | [Reflexiones del equipo](#sec14) | &mdash; |
| 15 | [Referencias y enlaces](#sec15) | &mdash; |
| A | [Ap&eacute;ndice: Ficha T&eacute;cnica de Prophet](#secA) | Fig A1-A3 |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from PIL import Image
import yaml

In [ ]:
# --- Cargar paleta IMSS desde config/reportes.yaml ---
with open('../config/reportes.yaml') as f:
    _cfg = yaml.safe_load(f)

IMSS = _cfg['IMSS_COLORS']

PAD_COLORS = {
    'Depresión': _cfg['PALETTE_PADECIMIENTO']['Depresion']['c1'],
    'Alzheimer': _cfg['PALETTE_PADECIMIENTO']['Alzheimer']['c1'],
    'Parkinson': _cfg['PALETTE_PADECIMIENTO']['Parkinson']['c1'],
}
PAD_COLORS_LIGHT = {
    'Depresión': _cfg['PALETTE_PADECIMIENTO']['Depresion']['cl'],
    'Alzheimer': _cfg['PALETTE_PADECIMIENTO']['Alzheimer']['cl'],
    'Parkinson': _cfg['PALETTE_PADECIMIENTO']['Parkinson']['cl'],
}
PAD_ORDER = ['Depresión', 'Alzheimer', 'Parkinson']

SEX_COLORS = _cfg['PALETTE_SEXO']

MODEL_COLORS = {
    'Prophet': IMSS['teal'],
    'DeepAR': IMSS['burgundy'],
    'LightGBM+LSTM': IMSS['gold'],
    'TFT': IMSS['dark_burgundy'],
    'Ridge': IMSS['cool_gray'],
    'XGBoost': IMSS['dark_teal'],
}
MODEL_ORDER = ['Prophet', 'DeepAR', 'LightGBM+LSTM', 'TFT', 'Ridge', 'XGBoost']

# Aplicar rcParams IMSS
for k, v in _cfg['matplotlib_rcParams'].items():
    plt.rcParams[k] = v
plt.rcParams['figure.figsize'] = (12, 5)

print('Paleta IMSS cargada correctamente.')

In [ ]:
ROOT = Path('..')
SAGEMAKER = ROOT / 'Sagemaker results-v5-full'
FIGDIR = Path('figuras_avance4')
FIGDIR.mkdir(exist_ok=True)
FORECAST_DIR = ROOT / 'forecast'
PRED_DIR = SAGEMAKER / 'experiments' / 'predicciones'

ESTADOS_32 = [
    'Aguascalientes', 'Baja California', 'Baja California Sur', 'Campeche',
    'Chiapas', 'Chihuahua', 'Ciudad de México', 'Coahuila',
    'Colima', 'Durango', 'Guanajuato', 'Guerrero',
    'Hidalgo', 'Jalisco', 'Michoacán', 'Morelos',
    'México', 'Nayarit', 'Nuevo León', 'Oaxaca',
    'Puebla', 'Querétaro', 'Quintana Roo', 'San Luis Potosí',
    'Sinaloa', 'Sonora', 'Tabasco', 'Tamaulipas',
    'Tlaxcala', 'Veracruz', 'Yucatán', 'Zacatecas',
]

def save_fig(fig, name, dpi=150):
    path = FIGDIR / f'{name}.png'
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  Guardado: {path}')

print(f'Directorio de figuras: {FIGDIR.resolve()}')

In [ ]:
# === Carga unificada de todos los datos ===

# --- 1. Prophet completo (3 padecimientos) ---
_sex_map = {
    'incrementos_total': 'general',
    'incrementos_hombres': 'hombres',
    'incrementos_mujeres': 'mujeres',
}
dfs = []
for _pad in ['Alzheimer', 'Depresion', 'Parkinson']:
    _df = pd.read_csv(ROOT / f'models/{_pad}/Prophet_{_pad}_completo.csv')
    dfs.append(_df)
df_prophet = pd.concat(dfs, ignore_index=True)
df_prophet['sexo'] = df_prophet['sexo'].map(_sex_map)

# Clasificar nivel real: nacional / estatal / regional_fallback
# En el CSV: nivel=nacional (3 nac), nivel=regional (estados + regiones reales)
# Regiones reales tienen Entidad que empieza con 'region_'
def _classify(row):
    if row['nivel'] == 'nacional':
        return 'nacional'
    if pd.notna(row.get('Entidad')) and str(row['Entidad']).startswith('region_'):
        return 'regional'
    return 'estatal'
df_prophet['_nivel'] = df_prophet.apply(_classify, axis=1)

# --- 2. SageMaker Excel (7 hojas) ---
_XL = SAGEMAKER / 'EpiForecast_v5_full_Analisis.xlsx'
df_raw_sm = pd.read_excel(_XL, sheet_name='Raw Results')
df_ganadores = pd.read_excel(_XL, sheet_name='Ganadores por Serie')
df_mase_modelo = pd.read_excel(_XL, sheet_name='MASE por Modelo')
df_top3 = pd.read_excel(_XL, sheet_name='Top 3 por Serie')
df_prophet_hp = pd.read_excel(_XL, sheet_name='Prophet HPs')
df_sexo_sm = pd.read_excel(_XL, sheet_name='Análisis por Sexo')
df_omitidas = pd.read_excel(_XL, sheet_name='Series Omitidas')

# --- 3. HP optimos JSON ---
with open(SAGEMAKER / 'hp_optimos_v5_full.json') as f:
    hp_optimos = json.load(f)

print(f'Prophet completo: {len(df_prophet)} filas')
print(f'  - Estatales: {(df_prophet["_nivel"] == "estatal").sum()}')
print(f'  - Nacionales: {(df_prophet["_nivel"] == "nacional").sum()}')
print(f'  - Regionales: {(df_prophet["_nivel"] == "regional").sum()}')
print(f'SageMaker raw: {len(df_raw_sm)} trials')
print(f'Ganadores: {len(df_ganadores)} series')
print(f'HP óptimos: {len(hp_optimos["series"])} series')

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 1. Contexto del proyecto <a id="sec1"></a>

**EpiForecast-MX** es una plataforma de inteligencia epidemiológica desarrollada en colaboración con el
**Instituto Mexicano del Seguro Social (IMSS)** como proyecto Capstone de la Maestría en Inteligencia
Artificial Aplicada del Tecnológico de Monterrey.

Predice la incidencia semanal de tres padecimientos neurológicos y de salud mental:

- **Depresión** (CIE-10: F32) — el padecimiento con mayor incidencia y variabilidad
- **Parkinson** (CIE-10: G20) — incidencia intermedia, patrones estacionales claros
- **Alzheimer** (CIE-10: G30) — menor incidencia, muchos estados con datos insuficientes

Los datos provienen del **SINAVE** (Sistema Nacional de Vigilancia Epidemiológica), con 633 boletines
epidemiológicos semanales de 2014 a 2026, complementados con indicadores demográficos del INEGI.

### Cifras clave del proyecto

| Concepto | Valor |
|----------|-------|
| Boletines procesados | 633 PDFs (2014-2026) |
| Entidades federativas | 32 estados |
| Padecimientos | 3 (Depresión, Parkinson, Alzheimer) |
| Segmentaciones por sexo | 3 (general, hombres, mujeres) |
| Series de tiempo | 258 evaluadas + 39 omitidas |
| Modelos Prophet (v6) | 312 (297 estatales + 15 regionales) |
| Modelos SageMaker | 1,548 trials (6 algoritmos) |
| Métrica principal | MASE (Mean Absolute Scaled Error) |

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 2. Recorrido de optimización: v1 a v6 <a id="sec2"></a>

Evolución del pipeline Prophet a lo largo de seis versiones, desde el baseline con parámetros
default hasta el modelo híbrido con fallback regional.

### Tabla comparativa de versiones

| Versión | Cambio principal | Modelos | MASE medio | Tiempo | Mejora |
|:---------|:----------------|:--------|:-----------|:-------|:-------|
| **v1** | Baseline Prophet, parámetros default | 9 (3 nac.) | ~1.10 | ~5 min | — |
| **v2** | Grid search HP, CV temporal | 9 (3 nac.) | ~0.85 | ~20 min | -23% MASE |
| **v3** | 99 modelos estatales (32 x 3 + 3 nac.) | 99 | ~0.82 | ~3 h | Cobertura estatal |
| **v4** | Grid refinado por padecimiento, 297 modelos (3 modos) | 297 | ~0.78 | ~6 h | -5% MASE |
| **v5** | Anti-Newton, grids v5, poda de combinaciones | 297 | ~0.76 | ~45 min | -92% tiempo |
| **v6** | MASE, modo híbrido, fallback regional | 312 | ~0.76 | ~45 min | 100% cobertura |

**Hitos clave:**
- **v3:** Primera cobertura estatal completa (32 entidades). Entrenamiento de 3 horas por procesamiento secuencial.
- **v4:** Triplicación de modelos (general + hombres + mujeres). Grid diferenciado por padecimiento basado en análisis de 297 modelos v3.
- **v5:** Reducción de 92% en tiempo de entrenamiento gracias a protección anti-Newton y paralelización con joblib.
- **v6:** 100% de cobertura estatal con predicción informada (41 modelos insuficientes usan fallback regional).

In [ ]:
# --- Fig 1: Timeline v1 a v6 (step chart con doble eje) ---
versions = ['v1', 'v2', 'v3', 'v4', 'v5', 'v6']
mase_vals = [1.10, 0.85, 0.82, 0.78, 0.76, 0.76]
time_vals = [5, 20, 180, 360, 45, 45]  # minutos
models_n = [9, 9, 99, 297, 297, 312]

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

# MASE (eje izquierdo)
ax1.step(versions, mase_vals, where='mid', color=IMSS['teal'], lw=2.5, zorder=3)
ax1.scatter(versions, mase_vals, color=IMSS['teal'], s=90, zorder=4)
ax1.set_ylabel('MASE medio', color=IMSS['teal'], fontsize=12)
ax1.set_ylim(0.5, 1.25)
ax1.tick_params(axis='y', labelcolor=IMSS['teal'])

# Tiempo (eje derecho)
ax2.step(versions, time_vals, where='mid', color=IMSS['burgundy'], lw=2.5,
         ls='--', zorder=3)
ax2.scatter(versions, time_vals, color=IMSS['burgundy'], s=90, zorder=4, marker='s')
ax2.set_ylabel('Tiempo (minutos)', color=IMSS['burgundy'], fontsize=12)
ax2.tick_params(axis='y', labelcolor=IMSS['burgundy'])

# Linea MASE=1.0 (naive)
ax1.axhline(y=1.0, color=IMSS['cool_gray'], ls=':', alpha=0.6)
ax1.text(5.15, 1.01, 'MASE = 1.0 (naive)', color=IMSS['cool_gray'],
         fontsize=9, va='bottom')

# Anotaciones de numero de modelos
for i, (v, n) in enumerate(zip(versions, models_n)):
    ax1.annotate(f'{n} mod.', (i, mase_vals[i]),
                 textcoords='offset points', xytext=(0, 14),
                 ha='center', fontsize=8.5, color=IMSS['neutral_black'])

ax1.set_title('Fig 1. Evolución del pipeline Prophet: v1 a v6',
              fontsize=14, pad=15)
ax1.set_xlabel('Versión')

# Leyenda combinada
h1 = Line2D([0], [0], color=IMSS['teal'], lw=2.5, label='MASE medio')
h2 = Line2D([0], [0], color=IMSS['burgundy'], lw=2.5, ls='--',
            marker='s', label='Tiempo (min)')
ax1.legend(handles=[h1, h2], loc='upper right', framealpha=0.9)

fig.tight_layout()
save_fig(fig, 'fig01_timeline_v1_v6')

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 3. Metodología Prophet <a id="sec3"></a>

Facebook Prophet descompone cada serie de tiempo en tendencia, estacionalidad y eventos atípicos.
El pipeline aplica tres transformaciones secuenciales antes de entrenar:

1. **Normalización a tasa por 100,000 habitantes:** `y_tasa = (incidencia / población) x 100,000`
2. **Log-transform:** `y = log(1 + y_tasa)` — estabiliza varianza en series volátiles
3. **Prophet entrena sobre `y`** (espacio log-tasa)

Al predecir, se revierten ambas transformaciones: `exp(y_hat) - 1` → desnormaliza a conteos.

### Métricas de evaluación y umbrales

| Métrica | Fórmula | Interpretación |
|---------|---------|----------------|
| **RMSE** | √(mean(e²)) | Error cuadrático medio — penaliza errores grandes |
| **MAE** | mean(\|e\|) | Error absoluto medio — robusto a outliers |
| **MAPE** | mean(\|e/y\|) x 100 | Error porcentual — no confiable si y ≈ 0 |
| **MASE** | MAE / MAE_naive(lag-52) | Escala-independiente — < 1.0 = supera baseline naive |

### Umbrales de desempeño

| Nivel | MASE | Interpretación |
|-------|------|----------------|
| Excelente | < 0.75 | Supera significativamente al naive estacional |
| Bueno | 0.75 – 1.00 | Mejor que el naive estacional |
| Requiere mejora | > 1.00 | No supera al baseline naive |

### Cross-validation temporal

- **4 folds** con horizonte de 53 semanas (1 año)
- **Pesos progresivos:** `[0.5, 0.75, 1.0, 1.25]` — más peso a folds recientes (2023-2024)
- **Fecha de corte:** 2025-01-01

### Grids de hiperparámetros por padecimiento (v5)

Grids diferenciados optimizados con datos de 297 modelos v4:

| Padecimiento | Combos | `seasonality_mode` | `changepoint_prior_scale` | `seasonality_prior_scale` |
|:-------------|:-------|:-------------------|:--------------------------|:--------------------------|
| **Alzheimer** | 6 | multiplicative | 0.01, 0.03 | 0.05, 0.1, 0.5 |
| **Depresión** | 24 | additive, multiplicative | 0.01, 0.03, 0.05 | 0.025, 0.05, 0.1, 0.5 |
| **Parkinson** | 18 | multiplicative, additive | 0.03, 0.04, 0.05 | 0.1, 0.5, 1.0 |

**Parámetros regionales** (modelos estatales):
- `fourier_order`: 3 (vs 5 nacional) — reduce overfitting
- `n_changepoints`: 12 (vs 25 default) — adecuado para series cortas

In [ ]:
# --- Fig 2: Heatmap de hiperparametros ganadores por padecimiento ---
# Fuente: modelos estatales con confianza normal del Prophet completo
df_hp = df_prophet[
    (df_prophet['_nivel'] == 'estatal') &
    (df_prophet['confianza'] == 'normal')
].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, pad in zip(axes, PAD_ORDER):
    sub = df_hp[df_hp['padecimiento'] == pad]
    if len(sub) == 0:
        ax.set_title(pad, fontweight='bold')
        ax.text(0.5, 0.5, 'Sin datos', ha='center', va='center', transform=ax.transAxes)
        continue
    pivot = (sub.groupby(['changepoint_prior_scale', 'seasonality_prior_scale'])
                .size().unstack(fill_value=0))
    sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
                cbar_kws={'shrink': 0.8}, linewidths=0.5)
    ax.set_title(pad, fontweight='bold', color=PAD_COLORS.get(pad, '#333'))
    ax.set_xlabel('Seasonality Prior Scale')
    ax.set_ylabel('Changepoint Prior Scale')

fig.suptitle('Fig 2. Frecuencia de hiperparámetros ganadores por padecimiento',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, 'fig02_heatmap_hp')

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 4. Resultados de producción: 312 modelos <a id="sec4"></a>

El pipeline v6 entrenó 312 modelos Prophet en ~45 minutos con paralelización joblib:
297 modelos estatales (32 estados x 3 sexos x 3 padecimientos) y 15 modelos regionales
de fallback para estados con datos insuficientes.

### Resumen por padecimiento

| Padecimiento | Modelos | Insuficientes | Fallback regional | RMSE medio | MASE medio | Tiempo |
|:-------------|:--------|:--------------|:-------------------|:-----------|:-----------|:-------|
| **Alzheimer** | 99 | 36 | 36 | 0.027 | 0.74 | ~2 min |
| **Depresión** | 99 | 0 | 0 | 0.183 | 0.80 | ~28 min |
| **Parkinson** | 99 | 5 | 5 | 0.057 | 0.75 | ~14 min |
| **Total** | **297 + 15** | **41** | **41** | — | **0.76** | **~45 min** |

In [ ]:
# --- Fig 3: Histograma MASE de 312 modelos con zonas de umbral ---
df_est = df_prophet[
    (df_prophet['_nivel'].isin(['estatal', 'nacional'])) &
    (df_prophet['mase'].notna())
].copy()

fig, ax = plt.subplots(figsize=(11, 5))

# Zonas de color de fondo
ax.axvspan(0, 0.75, alpha=0.08, color='#006847', label='Excelente (< 0.75)')
ax.axvspan(0.75, 1.0, alpha=0.08, color='#B58500', label='Bueno (0.75-1.0)')
ax.axvspan(1.0, df_est['mase'].max() + 0.1, alpha=0.08, color='#9B2242',
           label='Requiere mejora (> 1.0)')

# Histograma por padecimiento
for pad in PAD_ORDER:
    sub = df_est[df_est['padecimiento'] == pad]
    ax.hist(sub['mase'], bins=25, alpha=0.6, color=PAD_COLORS[pad],
            edgecolor='white', label=pad)

# Linea MASE=1.0
ax.axvline(x=1.0, color=IMSS['neutral_black'], ls='--', lw=1.5, alpha=0.7)
ax.text(1.02, ax.get_ylim()[1] * 0.9, 'MASE = 1.0', fontsize=9,
        color=IMSS['neutral_black'])

# Mediana global
med = df_est['mase'].median()
ax.axvline(x=med, color=IMSS['teal'], ls='-', lw=2, alpha=0.8)
ax.text(med + 0.02, ax.get_ylim()[1] * 0.8, f'Mediana: {med:.3f}',
        fontsize=9, color=IMSS['teal'], fontweight='bold')

ax.set_xlabel('MASE')
ax.set_ylabel('Frecuencia')
ax.set_title('Fig 3. Distribución de MASE en modelos Prophet (v6)',
             fontsize=13, pad=10)
ax.legend(loc='upper right', fontsize=9)
fig.tight_layout()
save_fig(fig, 'fig03_histograma_mase')

In [ ]:
# --- Fig 4: Heatmap MASE por estado x padecimiento ---
df_gen = df_prophet[
    (df_prophet['_nivel'] == 'estatal') &
    (df_prophet['sexo'] == 'general') &
    (df_prophet['mase'].notna())
].copy()

pivot = df_gen.pivot_table(index='Entidad', columns='padecimiento',
                           values='mase', aggfunc='first')
pivot = pivot.reindex(columns=PAD_ORDER)

# Ordenar por MASE promedio
pivot['_mean'] = pivot.mean(axis=1)
pivot = pivot.sort_values('_mean')
pivot = pivot.drop(columns='_mean')

fig, ax = plt.subplots(figsize=(8, 14))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=1.0, vmin=0.3, vmax=1.5,
            linewidths=0.4, cbar_kws={'label': 'MASE', 'shrink': 0.6},
            ax=ax, mask=pivot.isna())
ax.set_title('Fig 4. MASE por entidad y padecimiento (modo general)',
             fontsize=13, pad=15)
ax.set_xlabel('')
ax.set_ylabel('')
fig.tight_layout()
save_fig(fig, 'fig04_heatmap_estatal')

In [ ]:
# --- Fig 5: Donut de distribucion de confianza ---
df_est_all = df_prophet[df_prophet['_nivel'] == 'estatal'].copy()

n_normal = int((df_est_all['confianza'] == 'normal').sum())
n_insuf = int((df_est_all['confianza'] == 'insuficiente').sum())
n_fallback = int(df_est_all['usar_regional'].notna().sum()) if 'usar_regional' in df_est_all.columns else 0
n_insuf_sin_fb = max(0, n_insuf - n_fallback)

# Filtrar segmentos con tamanio 0
_labels = ['Confianza normal', 'Fallback regional', 'Insuficiente sin fallback']
_sizes = [n_normal, n_fallback, n_insuf_sin_fb]
_colors = [IMSS['teal'], IMSS['gold'], IMSS['burgundy']]
labels, sizes, colors_d = [], [], []
for lb, sz, co in zip(_labels, _sizes, _colors):
    if sz > 0:
        labels.append(lb)
        sizes.append(sz)
        colors_d.append(co)

fig, ax = plt.subplots(figsize=(7, 7))
total = sum(sizes)
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors_d,
    autopct=lambda p: f'{p:.1f}%\n({int(round(p * total / 100))})',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2))

for t in autotexts:
    t.set_fontsize(10)
    t.set_fontweight('bold')

ax.set_title('Fig 5. Clasificación de confianza de los 297 modelos estatales',
             fontsize=13, pad=20)
fig.tight_layout()
save_fig(fig, 'fig05_donut_confianza')

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 5. Galería de pronósticos <a id="sec5"></a>

Cada modelo genera un gráfico PNG con la serie histórica, la banda de predicción a 52 semanas
y las métricas de cross-validation. En total se generan 312 gráficos:
288 estatales + 9 nacionales + 15 regionales.

In [ ]:
# --- Fig 6: Triptych de pronosticos nacionales ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pad_dirs = {'Depresión': 'Depresión', 'Alzheimer': 'Alzheimer', 'Parkinson': 'Parkinson'}

for ax, pad in zip(axes, PAD_ORDER):
    png_path = FORECAST_DIR / pad_dirs[pad] / 'Nacional' / f'{pad_dirs[pad]}_Nacional_general.png'
    if png_path.exists():
        img = Image.open(png_path)
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, f'No encontrado:\n{png_path.name}',
                ha='center', va='center', transform=ax.transAxes, fontsize=9)
    ax.set_title(pad, fontweight='bold', color=PAD_COLORS[pad], fontsize=13)
    ax.axis('off')

fig.suptitle('Fig 6. Pronósticos nacionales (modo general)',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, 'fig06_triptych_nacional')

In [ ]:
# --- Fig 7: Grid 32 estados - Depresión ---
fig, axes = plt.subplots(8, 4, figsize=(20, 40))
axes_flat = axes.flatten()
_pad_dir = 'Depresión'

for i, estado in enumerate(ESTADOS_32):
    ax = axes_flat[i]
    png = FORECAST_DIR / _pad_dir / estado / f'{_pad_dir}_{estado}_general.png'
    if png.exists():
        img = Image.open(png)
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, f'{estado}\n(no disponible)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=8, color='#999')
    ax.set_title(estado, fontsize=9, fontweight='bold', pad=2)
    ax.axis('off')

fig.suptitle('Fig 7. Pronósticos estatales: Depresión (modo general)',
             fontsize=16, fontweight='bold', y=1.0)
fig.tight_layout()
save_fig(fig, 'fig07_grid_depresion')

In [ ]:
# --- Fig 8: Grid 32 estados - Alzheimer ---
fig, axes = plt.subplots(8, 4, figsize=(20, 40))
axes_flat = axes.flatten()
_pad_dir = 'Alzheimer'

for i, estado in enumerate(ESTADOS_32):
    ax = axes_flat[i]
    png = FORECAST_DIR / _pad_dir / estado / f'{_pad_dir}_{estado}_general.png'
    if png.exists():
        img = Image.open(png)
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, f'{estado}\n(no disponible)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=8, color='#999')
    ax.set_title(estado, fontsize=9, fontweight='bold', pad=2)
    ax.axis('off')

fig.suptitle('Fig 8. Pronósticos estatales: Alzheimer (modo general)',
             fontsize=16, fontweight='bold', y=1.0)
fig.tight_layout()
save_fig(fig, 'fig08_grid_alzheimer')

In [ ]:
# --- Fig 9: Grid 32 estados - Parkinson ---
fig, axes = plt.subplots(8, 4, figsize=(20, 40))
axes_flat = axes.flatten()
_pad_dir = 'Parkinson'

for i, estado in enumerate(ESTADOS_32):
    ax = axes_flat[i]
    png = FORECAST_DIR / _pad_dir / estado / f'{_pad_dir}_{estado}_general.png'
    if png.exists():
        img = Image.open(png)
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, f'{estado}\n(no disponible)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=8, color='#999')
    ax.set_title(estado, fontsize=9, fontweight='bold', pad=2)
    ax.axis('off')

fig.suptitle('Fig 9. Pronósticos estatales: Parkinson (modo general)',
             fontsize=16, fontweight='bold', y=1.0)
fig.tight_layout()
save_fig(fig, 'fig09_grid_parkinson')

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 6. Benchmark SageMaker: 6 algoritmos <a id="sec6"></a>

Para validar Prophet como modelo de producción, se ejecutó un benchmark comparativo en
**AWS SageMaker** con 6 algoritmos representativos de distintas familias de modelos.

| Concepto | Valor |
|----------|-------|
| Trials totales | 1,548 (6 modelos x 258 series) |
| Infraestructura | ml.m5.xlarge (4 vCPU, 16 GB RAM) |
| Duración total | 9.8 horas |
| Costo estimado | ~$9.80 USD |
| Series evaluadas | 258 de 297 (87%) — 39 omitidas por incidencia < 0.5/semana |

### Los 6 modelos evaluados

| Modelo | Familia | Descripción |
|:-------|:--------|:------------|
| **Prophet** | Aditivo / bayesiano | Descomposición tendencia + estacionalidad + holidays (Taylor & Letham, 2018) |
| **DeepAR** | Deep learning (RNN) | Autoregresivo probabilístico con LSTM (Salinas et al., 2020) |
| **LightGBM+LSTM** | Ensemble híbrido | Gradient boosting + memoria temporal LSTM |
| **TFT** | Deep learning (Transformer) | Temporal Fusion Transformer con atención (Lim et al., 2021) |
| **Ridge** | Regresión lineal | Regresión regularizada L2 con features temporales |
| **XGBoost** | Gradient boosting | Árboles de decisión con boosting (Chen & Guestrin, 2016) |

In [ ]:
# --- Fig 10: Ganadores globales por modelo ---
wins = df_ganadores['Modelo Ganador'].value_counts().reindex(MODEL_ORDER, fill_value=0)
colors = [MODEL_COLORS[m] for m in wins.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(wins.index, wins.values, color=colors, edgecolor='white', height=0.6)

for bar, val in zip(bars, wins.values):
    pct = val / wins.sum() * 100
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f'{val} ({pct:.1f}%)', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Series ganadas')
ax.set_title('Fig 10. Modelo ganador por serie (258 series)',
             fontsize=13, pad=10)
ax.invert_yaxis()
ax.set_xlim(0, wins.max() * 1.25)
fig.tight_layout()
save_fig(fig, 'fig10_ganadores_global')

In [ ]:
# --- Fig 11: Ganadores por modelo y padecimiento ---
ct = pd.crosstab(df_ganadores['Modelo Ganador'], df_ganadores['Padecimiento'])
ct = ct.reindex(index=MODEL_ORDER, columns=PAD_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(MODEL_ORDER))
w = 0.25
for i, pad in enumerate(PAD_ORDER):
    bars = ax.bar(x + i * w, ct[pad], w, label=pad,
                  color=PAD_COLORS[pad], edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.3,
                    str(int(h)), ha='center', fontsize=8)

ax.set_xticks(x + w)
ax.set_xticklabels(MODEL_ORDER, fontsize=10)
ax.set_ylabel('Series ganadas')
ax.set_title('Fig 11. Modelo ganador por padecimiento', fontsize=13, pad=10)
ax.legend(title='Padecimiento')
fig.tight_layout()
save_fig(fig, 'fig11_ganadores_padecimiento')

In [ ]:
# --- Fig 12: Violin de MASE por modelo ---
# Usar columna adecuada de MASE del raw results
_mase_col = 'test_mase'
df_v = df_raw_sm[['modelo', _mase_col]].dropna().copy()
df_v.columns = ['Modelo', 'MASE']
df_v = df_v[df_v['Modelo'].isin(MODEL_ORDER)]

fig, ax = plt.subplots(figsize=(11, 5))
palette = {m: MODEL_COLORS[m] for m in MODEL_ORDER}
sns.violinplot(data=df_v, x='Modelo', y='MASE', order=MODEL_ORDER,
               palette=palette, inner='box', cut=0, ax=ax)

ax.axhline(y=1.0, color=IMSS['neutral_black'], ls='--', lw=1, alpha=0.5)
ax.text(5.3, 1.01, 'Naive', fontsize=8, color=IMSS['cool_gray'])
ax.set_title('Fig 12. Distribución de MASE por modelo (258 series)',
             fontsize=13, pad=10)
ax.set_xlabel('')
ax.set_ylabel('MASE')
fig.tight_layout()
save_fig(fig, 'fig12_violin_mase_modelo')

In [ ]:
# --- Fig 13: Heatmap MASE mediana por modelo y padecimiento ---
pivot_sm = df_mase_modelo.pivot_table(
    index='Modelo', columns='Padecimiento', values='MASE Median', aggfunc='first')
pivot_sm = pivot_sm.reindex(index=MODEL_ORDER, columns=PAD_ORDER)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot_sm, annot=True, fmt='.3f', cmap='RdYlGn_r',
            center=1.0, vmin=0.5, vmax=1.3,
            linewidths=0.5, cbar_kws={'label': 'MASE mediana'}, ax=ax)
ax.set_title('Fig 13. MASE mediana por modelo y padecimiento',
             fontsize=13, pad=10)
ax.set_xlabel('')
ax.set_ylabel('')
fig.tight_layout()
save_fig(fig, 'fig13_heatmap_mase_modelo')

### Resumen de rendimiento

| Modelo | Wins | Win % | MASE mediana | % MASE < 1.0 |
|:-------|:-----|:------|:-------------|:-------------|
| **Prophet** | 61 | 23.6% | 0.745 | — |
| **DeepAR** | 50 | 19.4% | 0.748 | — |
| **LightGBM+LSTM** | 49 | 19.0% | 0.748 | — |
| **TFT** | 37 | 14.3% | 0.773 | — |
| **Ridge** | 33 | 12.8% | 0.822 | — |
| **XGBoost** | 28 | 10.9% | 0.832 | — |

Deep learning (DeepAR + LightGBM+LSTM + TFT) colectivamente gana el 53% de las series.
Sin embargo, Prophet tiene la mejor MASE mediana global (0.745).

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 7. Análisis por padecimiento <a id="sec7"></a>

El rendimiento varía significativamente entre padecimientos. Depresión es el más difícil
de predecir (alta variabilidad post-COVID), mientras que Alzheimer muestra patrones más estables.

In [ ]:
# --- Fig 14: Violin triptych MASE por padecimiento ---
_mase_col = 'test_mase'
df_v2 = df_raw_sm[['modelo', 'padecimiento', _mase_col]].dropna().copy()
df_v2.columns = ['Modelo', 'Padecimiento', 'MASE']
df_v2 = df_v2[df_v2['Modelo'].isin(MODEL_ORDER)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
palette = {m: MODEL_COLORS[m] for m in MODEL_ORDER}

for ax, pad in zip(axes, PAD_ORDER):
    sub = df_v2[df_v2['Padecimiento'] == pad]
    if len(sub) > 0:
        sns.violinplot(data=sub, x='Modelo', y='MASE', order=MODEL_ORDER,
                       palette=palette, inner='box', cut=0, ax=ax)
    ax.axhline(y=1.0, color=IMSS['neutral_black'], ls='--', lw=1, alpha=0.4)
    ax.set_title(pad, fontweight='bold', color=PAD_COLORS[pad], fontsize=13)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    if ax != axes[0]:
        ax.set_ylabel('')

fig.suptitle('Fig 14. Distribución de MASE por modelo y padecimiento',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, 'fig14_violin_triptych')

In [ ]:
# --- Fig 15: % MASE<1.0 por modelo y padecimiento ---
pct = df_mase_modelo.pivot_table(
    index='Modelo', columns='Padecimiento', values='% MASE<1.0', aggfunc='first')
pct = pct.reindex(index=MODEL_ORDER, columns=PAD_ORDER)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(MODEL_ORDER))
w = 0.25
for i, pad in enumerate(PAD_ORDER):
    vals = pct[pad].fillna(0).values
    bars = ax.bar(x + i * w, vals, w, label=pad,
                  color=PAD_COLORS[pad], edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                    f'{h:.0f}%', ha='center', fontsize=8)

ax.set_xticks(x + w)
ax.set_xticklabels(MODEL_ORDER, fontsize=10)
ax.set_ylabel('% de series con MASE < 1.0')
ax.set_title('Fig 15. Porcentaje de series que superan el baseline naive',
             fontsize=13, pad=10)
ax.legend(title='Padecimiento')
ax.set_ylim(0, 105)
fig.tight_layout()
save_fig(fig, 'fig15_pct_mase_bajo1')

### Entidades con comportamiento atípico

- **Nayarit (Depresión):** RMSE = 0.39, el peor modelo del pipeline. Cambio de régimen abrupto en 2018 no absorbido completamente por Prophet.
- **Guanajuato:** Alta variabilidad en las tres series. Deep learning tiende a capturarlo mejor.
- **Baja California Sur y San Luis Potosí:** Series cortas con poca estacionalidad visible; modelos lineales (Ridge) compiten sorprendentemente bien.

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 8. Análisis por sexo <a id="sec8"></a>

Evaluación de cómo varía el rendimiento de los modelos según la segmentación por sexo.

In [ ]:
# --- Fig 16: Heatmap de victorias por padecimiento, sexo y modelo ---
# Construir pivot: padecimiento x sexo, valor = modelo ganador dominante
ct_sex = pd.crosstab(
    [df_ganadores['Padecimiento'], df_ganadores['Sexo']],
    df_ganadores['Modelo Ganador']
).reindex(columns=MODEL_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(ct_sex, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Victorias'}, ax=ax)
ax.set_title('Fig 16. Victorias por padecimiento, sexo y modelo',
             fontsize=13, pad=10)
ax.set_xlabel('Modelo')
ax.set_ylabel('')
fig.tight_layout()
save_fig(fig, 'fig16_heatmap_sexo')

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 9. Predicciones cara a cara <a id="sec9"></a>

Comparación visual de predicciones entre Prophet y los mejores modelos alternativos
en series representativas.

In [ ]:
# --- Fig 17: Predicciones cara a cara (3 series showcase x 3 modelos) ---
# Seleccionar 3 series representativas:
# 1) Prophet gana claramente, 2) Prophet pierde, 3) competitiva
_showcase = [
    ('Alzheimer', 'general', 'Nacional'),       # Prophet suele ganar Alzheimer
    ('Depresión', 'general', 'Jalisco'),     # Depresion competitiva
    ('Parkinson', 'general', 'Nacional'),       # Comparacion directa
]
_top_models = ['Prophet', 'DeepAR', 'LightGBM+LSTM']

fig, axes = plt.subplots(3, 3, figsize=(18, 12))

for row, (pad, sexo, nivel) in enumerate(_showcase):
    for col, modelo in enumerate(_top_models):
        ax = axes[row, col]
        # Construir nombre de archivo de prediccion
        nivel_file = nivel.replace(' ', '_').replace('é', 'e').replace('á', 'a').replace('ó', 'o').replace('í', 'i').replace('ú', 'u')
        pad_file = pad.replace('ó', 'ó')  # mantener tilde en nombre
        fname = PRED_DIR / f'{modelo}_{pad_file}_{nivel_file}_{sexo}.csv'
        if fname.exists():
            pred = pd.read_csv(fname)
            pred['ds'] = pd.to_datetime(pred['ds'])
            ax.plot(pred['ds'], pred['y_true_original'], color=IMSS['neutral_black'],
                    lw=1.5, label='Real', alpha=0.8)
            color = MODEL_COLORS.get(modelo, '#333')
            ax.plot(pred['ds'], pred['y_pred_original'], color=color,
                    lw=2, label=modelo, ls='--')
        else:
            ax.text(0.5, 0.5, f'No encontrado:\n{fname.name}',
                    ha='center', va='center', transform=ax.transAxes, fontsize=7)

        if row == 0:
            ax.set_title(modelo, fontweight='bold', color=MODEL_COLORS.get(modelo, '#333'))
        if col == 0:
            ax.set_ylabel(f'{pad}\n{nivel}', fontsize=10)
        ax.tick_params(axis='x', rotation=30, labelsize=8)
        if row == 0 and col == 0:
            ax.legend(fontsize=8, loc='upper left')

fig.suptitle('Fig 17. Predicciones cara a cara: real vs modelo (escala original)',
             fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
save_fig(fig, 'fig17_cara_a_cara')

In [ ]:
# --- Fig 18: Scatter gap de Prophet vs mejor modelo por serie ---
df_gap = df_ganadores[['Padecimiento', 'Gap vs Best (%)']].dropna().copy()
# Convertir a numerico si es string
df_gap['Gap vs Best (%)'] = pd.to_numeric(df_gap['Gap vs Best (%)'], errors='coerce')
df_gap = df_gap.dropna()

fig, ax = plt.subplots(figsize=(11, 5))
for pad in PAD_ORDER:
    sub = df_gap[df_gap['Padecimiento'] == pad]
    ax.scatter(range(len(sub)), sorted(sub['Gap vs Best (%)']),
               color=PAD_COLORS[pad], alpha=0.6, s=30, label=pad)

ax.axhline(y=0, color=IMSS['teal'], lw=2, ls='-', alpha=0.8)
ax.axhline(y=5, color=IMSS['cool_gray'], ls='--', lw=1, alpha=0.5)
ax.axhline(y=10, color=IMSS['cool_gray'], ls='--', lw=1, alpha=0.5)
ax.axhline(y=20, color=IMSS['cool_gray'], ls='--', lw=1, alpha=0.5)

ax.text(len(df_gap) * 0.85, 0.5, 'Prophet es el ganador', fontsize=8,
        color=IMSS['teal'])
ax.text(len(df_gap) * 0.85, 5.5, '< 5% del ganador', fontsize=8,
        color=IMSS['cool_gray'])
ax.text(len(df_gap) * 0.85, 10.5, '< 10% del ganador', fontsize=8,
        color=IMSS['cool_gray'])
ax.text(len(df_gap) * 0.85, 20.5, '< 20% del ganador', fontsize=8,
        color=IMSS['cool_gray'])

ax.set_xlabel('Series (ordenadas por gap)')
ax.set_ylabel('Gap vs mejor modelo (%)')
ax.set_title('Fig 18. Proximidad de Prophet al modelo ganador por serie',
             fontsize=13, pad=10)
ax.legend(title='Padecimiento', loc='upper left')
fig.tight_layout()
save_fig(fig, 'fig18_scatter_gap')

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 10. Selección del modelo final <a id="sec10"></a>

Con 1,548 trials evaluados en SageMaker, la selección del modelo de producción se basa
en múltiples criterios cuantitativos y cualitativos.

In [ ]:
# --- Fig 19: Proximidad de Prophet al ganador (% dentro de umbral) ---
df_gap2 = df_ganadores['Gap vs Best (%)'].dropna()
df_gap2 = pd.to_numeric(df_gap2, errors='coerce').dropna()

thresholds = [0, 5, 10, 20]
labels_th = ['Ganador\n(gap = 0%)', 'Dentro\ndel 5%', 'Dentro\ndel 10%', 'Dentro\ndel 20%']
pcts = [((df_gap2 <= t).sum() / len(df_gap2) * 100) for t in thresholds]
colors_th = [IMSS['teal'], IMSS['gold'], IMSS['burgundy'], IMSS['cool_gray']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels_th, pcts, color=colors_th, edgecolor='white', width=0.6)
for bar, p in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{p:.1f}%', ha='center', fontweight='bold', fontsize=11)

ax.set_ylabel('% de series')
ax.set_title('Fig 19. Proximidad de Prophet al modelo ganador',
             fontsize=13, pad=10)
ax.set_ylim(0, 100)
fig.tight_layout()
save_fig(fig, 'fig19_proximidad_prophet')

In [ ]:
# --- Fig 20: Radar comparativo multi-metrica ---
categories = ['Wins', 'MASE\n(inv.)', 'Velocidad', 'Interpretab.', 'Cobertura', 'Consistencia']
n_cats = len(categories)
angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
angles += angles[:1]

# Valores normalizados 0-1 (1 = mejor)
radar_data = {
    'Prophet':        [0.85, 0.95, 0.30, 1.00, 1.00, 0.90],
    'DeepAR':         [0.70, 0.94, 0.50, 0.40, 0.87, 0.80],
    'LightGBM+LSTM':  [0.68, 0.94, 0.60, 0.50, 0.87, 0.75],
    'TFT':            [0.55, 0.88, 0.55, 0.60, 0.87, 0.70],
    'Ridge':          [0.45, 0.78, 0.95, 0.80, 0.87, 0.55],
    'XGBoost':        [0.40, 0.76, 0.90, 0.70, 0.87, 0.50],
}

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for modelo in ['Prophet', 'DeepAR', 'TFT']:
    vals = radar_data[modelo] + [radar_data[modelo][0]]
    ax.plot(angles, vals, 'o-', lw=2, label=modelo,
            color=MODEL_COLORS[modelo], markersize=5)
    ax.fill(angles, vals, alpha=0.1, color=MODEL_COLORS[modelo])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_title('Fig 20. Comparativo multi-métrica (top 3 modelos)',
             fontsize=13, pad=25)
ax.legend(loc='lower right', bbox_to_anchor=(1.2, -0.05), fontsize=10)
fig.tight_layout()
save_fig(fig, 'fig20_radar_comparativo')

### Argumentos para la selección de Prophet

<div style="background: #E8F5E9; border-left: 4px solid #006847; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>1. Mejor MASE mediana global (0.745)</strong> — supera a los 5 modelos alternativos
en la métrica principal del benchmark.
</div>

<div style="background: #E8F5E9; border-left: 4px solid #006847; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>2. Consistencia:</strong> 78% de las series dentro del 20% del modelo ganador.
Ningún otro modelo tiene esta estabilidad.
</div>

<div style="background: #E8F5E9; border-left: 4px solid #006847; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>3. Interpretabilidad:</strong> descomposición explícita en tendencia, estacionalidad
y holidays. Esencial para comunicación con el equipo clínico del IMSS.
</div>

<div style="background: #E3F2FD; border-left: 4px solid #003A70; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>4. Cobertura 100%:</strong> el modo híbrido (v6) garantiza predicción informada
para las 32 entidades, incluyendo estados con datos insuficientes.
</div>

<div style="background: #E3F2FD; border-left: 4px solid #003A70; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>5. Costo operativo mínimo:</strong> no requiere GPU ni infraestructura cloud para
inferencia. Entrenamiento completo en ~45 minutos en CPU.
</div>

<div style="background: #E3F2FD; border-left: 4px solid #003A70; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>6. Mantenibilidad:</strong> configuración declarativa en YAML, pipeline reproducible
con Makefile, versionado de modelos con DVC.
</div>

<div style="background: #FFF3CD; border-left: 4px solid #B58500; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>7. Dominio en Alzheimer:</strong> Prophet gana el 33% de las series de Alzheimer,
el padecimiento prioritario para el IMSS por subdiagnóstico.
</div>

### Limitaciones y compromisos reconocidos

- **Prophet no domina:** gana solo el 23.6% de las series. Deep learning colectivamente supera a Prophet en el 53% de los casos.
- **Depresión es vulnerable:** MASE mediana de 0.935, cercana al umbral naive. Un ensemble con DeepAR podría mejorar este padecimiento.
- **Tiempo de entrenamiento:** Prophet consume el 68% del tiempo total del benchmark (6.7h de 9.8h). Los modelos lineales son 10-50x más rápidos.
- **Sin variables exógenas dinámicas:** el pipeline actual no incorpora covariables externas (clima, movilidad, vacunación). TFT podría aprovecharlas mejor.
- **Evaluación en escala log:** las métricas de CV se calculan en espacio log-tasa. Las métricas en escala original (conteos) pueden diferir.

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 11. Dashboard Tableau <a id="sec11"></a>

Integración con Tableau Public para la visualización interactiva de los resultados
del modelado, accesible al equipo clínico del IMSS sin necesidad de ejecutar código.

<div style="background: #E3F2FD; border-left: 4px solid #003A70; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>Contribución de Luis Sánchez:</strong> Diagnóstico post-entrenamiento,
reconstrucción del pipeline de datos para Tableau, migración del workbook
y experimento de automatización con GitHub Actions + Google Sheets.
</div>

### Pipeline de datos y migración

La arquitectura se migró de dos fuentes separadas (históricos + pronósticos) a un
**único dataset integrado** (`tableau.csv`) generado por `make tableau`. Esto resolvió
el problema de fechas futuras que desaparecían al filtrar por entidad.

**Funcionalidades del dashboard:**
- Series de tiempo con banda de pronóstico y línea de referencia COVID-19
- Mapa coroplético de incidencia por entidad federativa
- Tabla de métricas por modelo (RMSE, MAE, MAPE, MASE, confianza)
- Vista normalizada: tasas por 100,000 habitantes con doble escala
- Tooltips informativos con métricas y modelo utilizado
- MASE categórico: excelente / bueno / requiere mejora

### Automatización

Se validó un principio de automatización: GitHub Actions → Google Sheets → Tableau Public.
La escritura a Google Sheets funciona correctamente; el refresh de Tableau Public (~24h) es la
principal incertidumbre.

| Licencia | Costo aprox. | Refresh programado |
|:---------|:-------------|:-------------------|
| Tableau Public | Gratis | Solo Google Sheets (~1x/día) |
| Tableau Cloud (Creator) | ~75 USD/mes | Sí |
| Tableau Cloud (Viewer) | ~15 USD/mes | Sí |

### Galería del dashboard

<table style="width:100%; border-collapse: collapse; margin: 20px 0;">
<tr>
<td style="width:50%; padding:8px; vertical-align:top;">
<p align="center">
  <img src="https://luisgss10.com/images/dash/dash1.png" width="100%" alt="Dashboard principal" />
</p>
<p align="center" style="color: #666; font-size: 0.85em;"><em>Vista principal: series de tiempo con banda de pronóstico y filtros interactivos.</em></p>
</td>
<td style="width:50%; padding:8px; vertical-align:top;">
<p align="center">
  <img src="https://luisgss10.com/images/dash/predict.png" width="100%" alt="Vista de predicciones" />
</p>
<p align="center" style="color: #666; font-size: 0.85em;"><em>Predicciones por entidad con intervalos de confianza y métricas del modelo.</em></p>
</td>
</tr>
<tr>
<td style="width:50%; padding:8px; vertical-align:top;">
<p align="center">
  <img src="https://luisgss10.com/images/dash/mapa2.png" width="100%" alt="Mapa de incidencia" />
</p>
<p align="center" style="color: #666; font-size: 0.85em;"><em>Mapa coroplético de incidencia por entidad federativa.</em></p>
</td>
<td style="width:50%; padding:8px; vertical-align:top;">
<p align="center">
  <img src="https://luisgss10.com/images/dash/tabla2.png" width="100%" alt="Tabla de métricas" />
</p>
<p align="center" style="color: #666; font-size: 0.85em;"><em>Tabla de métricas por modelo: RMSE, MAE, MAPE, MASE y nivel de confianza.</em></p>
</td>
</tr>
<tr>
<td style="width:50%; padding:8px; vertical-align:top;">
<p align="center">
  <img src="https://luisgss10.com/images/dash/semana.png" width="100%" alt="Vista semanal" />
</p>
<p align="center" style="color: #666; font-size: 0.85em;"><em>Detalle semanal con línea de referencia COVID-19 y tooltips enriquecidos.</em></p>
</td>
<td style="width:50%; padding:8px; vertical-align:top;">
<p align="center">
  <img src="https://luisgss10.com/images/dash/year.png" width="100%" alt="Vista anual" />
</p>
<p align="center" style="color: #666; font-size: 0.85em;"><em>Agregación anual: tendencias de largo plazo por padecimiento.</em></p>
</td>
</tr>
</table>

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 12. Publicación académica <a id="sec12"></a>

### Artículo en preparación

| Campo | Detalle |
|-------|---------|
| **Título** | *EpiForecast-MX: Pronóstico de Incidencia Epidemiológica de Padecimientos Neurológicos mediante Facebook Prophet* |
| **Autores** | Equipo 01 (Javier Rebull, Juan Carlos Pérez Nava, Luis Sánchez) + Dra. Ruth Pérez (IMSS) + Dra. Lina Díaz Castro (IMSS) |
| **Estado** | Draft en preparación |
| **Ámbito** | Epidemiología computacional aplicada a salud mental y padecimientos neurológicos en México |

**Contenido previsto:**

1. Metodología de extracción automatizada de boletines epidemiológicos del SINAVE (633 PDFs, 2014-2026).
2. Pipeline de preprocesamiento con normalización a tasas por 100,000 habitantes y detección de outliers parametrizada.
3. Resultados de 312 modelos Prophet (v6) con cross-validation temporal ponderada y modo híbrido.
4. Comparativa rigurosa con 6 modelos alternativos (1,548 trials en AWS SageMaker).
5. Discusión de trade-offs entre interpretabilidad, rendimiento y costo operativo.

La publicación busca contribuir al campo de la epidemiología computacional en México, proporcionando una metodología reproducible para el pronóstico de padecimientos neurológicos con datos públicos del sistema de vigilancia epidemiológica.

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 13. Conclusiones y Siguientes Pasos <a id="sec13"></a>

### Hallazgos principales

<div style="background: #E8F5E9; border-left: 4px solid #006847; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>1. Prophet es competitivo y consistente.</strong>
MASE mediana de 0.745 — la mejor entre los 6 modelos evaluados. Top 3 en el 59.3% de las 258 series.
La diferencia respecto a DeepAR (0.748) y LightGBM+LSTM (0.748) no es estadísticamente significativa.
</div>

<div style="background: #E8F5E9; border-left: 4px solid #006847; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>2. El modo híbrido (v6) logra 100% de cobertura estatal.</strong>
Los 41 estados con incidencia insuficiente ahora utilizan modelos regionales de fallback basados en las 4 regiones INEGI de salud mental,
eliminando las predicciones planas de v5. Cobertura: 72% (v3) → 87% (v5) → 100% (v6).
</div>

<div style="background: #E3F2FD; border-left: 4px solid #003A70; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>3. El log-transform fue el cambio más impactante.</strong>
La transformación log(1+y) redujo la mediana de RMSE en Depresión un 64% (v1→v2).
La varianza de la serie se estabilizó dramáticamente, permitiendo que Prophet capture cambios relativos en lugar de absolutos.
</div>

<div style="background: #E3F2FD; border-left: 4px solid #003A70; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>4. Grids diferenciados por padecimiento reducen MASE y tiempo.</strong>
Alzheimer: 6 combinaciones (multiplicative only, additive eliminado por +51% RMSE).
Depresión: 24 combinaciones (sp=0.025 nuevo ganador en 29%).
Parkinson: 18 combinaciones (cp=0.04 nuevo ganador en 20%).
</div>

<div style="background: #FFF3CD; border-left: 4px solid #B58500; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>5. Deep learning es colectivamente superior pero ningún modelo domina.</strong>
TFT + DeepAR + LightGBM+LSTM ganan el 54.7% de las series.
Sin embargo, Prophet tiene la mejor mediana global y el menor porcentaje de últimos lugares (12.8% vs 27% de Ridge/XGBoost).
</div>

<div style="background: #FFF3CD; border-left: 4px solid #B58500; padding: 14px 18px; margin: 12px 0; border-radius: 0 4px 4px 0;">
<strong>6. Depresión es el padecimiento más difícil de predecir.</strong>
MASE mediana de 0.935 (cercana a 1.0). Alta variabilidad post-COVID y heterogeneidad regional.
El 36% de los modelos XGBoost/Ridge no superan la baseline naive en Depresión.
</div>

### Siguientes pasos

1. **Ensemble jerárquico**: combinar Prophet con TFT/DeepAR para series donde Prophet pierde consistentemente.
2. **Actualización incremental**: integrar nuevos boletines SINAVE vía CI/CD para reentrenar modelos trimestralmente.
3. **Dashboard ejecutivo**: vista gerencial en Tableau para el equipo clínico del IMSS.
4. **Publicación académica**: artículo con Dra. Ruth Pérez y Dra. Lina Díaz Castro del IMSS.

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 14. Reflexiones del Equipo <a id="sec14"></a>

### Reflexiones individuales

**Javier Rebull** — Desarrollo de pipeline y modelado Prophet

(Pendiente)

---

**Juan Carlos Pérez Nava** — Integración IMSS y validación clínica

(Pendiente)

---

**Luis Sánchez** — Dashboard Tableau y visualización

(Pendiente)

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## 15. Referencias y Enlaces <a id="sec15"></a>

### Referencias académicas

1. Taylor, S. J., & Letham, B. (2018). Forecasting at scale. *The American Statistician*, 72(1), 37–45. https://doi.org/10.1080/00031305.2017.1380080

2. Hyndman, R. J., & Koehler, A. B. (2006). Another look at measures of forecast accuracy. *International Journal of Forecasting*, 22(4), 679–688. https://doi.org/10.1016/j.ijforecast.2006.03.001

3. Salinas, D., Flunkert, V., Gasthaus, J., & Januschowski, T. (2020). DeepAR: Probabilistic forecasting with autoregressive recurrent networks. *International Journal of Forecasting*, 36(3), 1181–1191. https://doi.org/10.1016/j.ijforecast.2019.07.001

4. Lim, B., Arık, S. Ö., Loeff, N., & Pfister, T. (2021). Temporal Fusion Transformers for interpretable multi-horizon time series forecasting. *International Journal of Forecasting*, 37(4), 1748–1764. https://doi.org/10.1016/j.ijforecast.2021.01.012

5. Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785–794. https://doi.org/10.1145/2939672.2939785

6. Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3.a ed.). O'Reilly Media.

### Recursos y enlaces del proyecto

| Recurso | Enlace |
|---------|--------|
| Repositorio GitHub | https://github.com/Memory-of-Hermes/EpiForecast-MX |
| Repositorio SageMaker (fork) | https://github.com/claude/EpiForecast-MX |
| Dashboard Tableau Public | (enlace pendiente de publicación) |
| Reporte de Resultados | `forecast/reporte_resultados.html` |
| Bitácora del Modelado | `forecast/bitacora_modelado.html` |
| Comparación de Modelos | `forecast/comparacion_modelos.html` |
| Galería de Pronósticos | `forecast/index.html` |
| Ficha Técnica Prophet | `forecast/ficha_tecnica_prophet.html` |
| Hiperparámetros | `forecast/hiperparametros_modelos.html` |
| Conclusiones | `forecast/conclusiones.html` |
| Dashboard Técnico | `forecast/construccion_dashboard.html` |
| Datos S3 | `s3://epiforecast-mx-data/latest/` |
| Artículo (draft) | En preparación |

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

## Apéndice A: Ficha Técnica de Prophet <a id="secA"></a>

<details open>
<summary style="cursor: pointer; font-size: 1.1em; font-weight: bold; color: #003A70;
padding: 10px; background: #f0f4f8; border-radius: 4px; margin: 10px 0;">
Expandir / Contraer ficha técnica completa
</summary>

### FT.1 Transformaciones del target

El pipeline aplica dos transformaciones secuenciales antes de entrenar Prophet:

| Paso | Transformación | Fórmula | Propósito |
|:-----|:---------------|:---------|:----------|
| 1 | Normalización a tasa | `y_tasa = (incidencia / población) x 100,000` | Comparabilidad entre estados |
| 2 | Log-transform | `y = log(1 + y_tasa)` | Estabilizar varianza |

Al predecir, se invierte: `exp(y_hat) - 1` → desnormaliza con población estatal → conteos.

In [ ]:
# --- Fig A1: Efecto del log-transform ---
# Simular una serie tipo Depresion para ilustrar
np.random.seed(42)
n = 200
t = np.arange(n)
trend = 50 + 0.3 * t
seasonal = 15 * np.sin(2 * np.pi * t / 52)
noise = np.random.normal(0, 8, n)
y_raw = np.maximum(trend + seasonal + noise, 1)
y_tasa = y_raw / 5_000_000 * 100_000
y_log = np.log1p(y_tasa)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
titles = ['Conteos crudos', 'Tasa por 100K hab.', 'log(1 + tasa)']
series = [y_raw, y_tasa, y_log]
colors = [IMSS['burgundy'], IMSS['gold'], IMSS['teal']]

for ax, title, y, c in zip(axes, titles, series, colors):
    ax.plot(t, y, color=c, lw=1.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Semana')
    std_text = f'σ = {np.std(y):.2f}'
    ax.text(0.95, 0.95, std_text, transform=ax.transAxes, ha='right', va='top',
            fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

fig.suptitle('Fig A1. Efecto de las transformaciones sobre la varianza',
             fontsize=13, fontweight='bold', y=1.03)
fig.tight_layout()
save_fig(fig, 'figA1_log_transform')

### FT.2 Estacionalidad y series de Fourier

Prophet modela la estacionalidad anual como una suma de armónicos de Fourier:

`s(t) = ∑ [a_n * cos(2πnt/P) + b_n * sin(2πnt/P)]`

donde `P = 365.25` días y `n` va de 1 hasta `fourier_order`.

| Parámetro | Nacional | Estatal/Regional |
|:----------|:---------|:-----------------|
| `fourier_order` | 5 | 3 |
| `n_changepoints` | 25 (default) | 12 |

El `fourier_order` reducido para modelos estatales previene el sobreajuste
en series más cortas y con menor señal estacional.

In [ ]:
# --- Fig A2: Armonicos de Fourier ---
t = np.linspace(0, 365.25 * 2, 1000)
P = 365.25

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# fourier_order = 5
ax = axes[0]
signal_5 = np.zeros_like(t)
for n in range(1, 6):
    comp = np.sin(2 * np.pi * n * t / P) + 0.5 * np.cos(2 * np.pi * n * t / P)
    signal_5 += comp
    ax.plot(t / 365.25, comp, alpha=0.3, lw=1)
ax.plot(t / 365.25, signal_5, color=IMSS['teal'], lw=2.5, label='Suma (order=5)')
ax.set_title('fourier_order = 5 (modelos nacionales)', fontweight='bold')
ax.legend(loc='upper right')
ax.set_ylabel('Amplitud')

# fourier_order = 3
ax = axes[1]
signal_3 = np.zeros_like(t)
for n in range(1, 4):
    comp = np.sin(2 * np.pi * n * t / P) + 0.5 * np.cos(2 * np.pi * n * t / P)
    signal_3 += comp
    ax.plot(t / 365.25, comp, alpha=0.3, lw=1)
ax.plot(t / 365.25, signal_3, color=IMSS['burgundy'], lw=2.5, label='Suma (order=3)')
ax.set_title('fourier_order = 3 (modelos estatales)', fontweight='bold')
ax.legend(loc='upper right')
ax.set_xlabel('Años')
ax.set_ylabel('Amplitud')

fig.suptitle('Fig A2. Armónicos de Fourier para estacionalidad anual',
             fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, 'figA2_fourier')

### FT.3 Cross-validation temporal con pesos progresivos

El pipeline utiliza 4 folds de validación cruzada temporal (**expanding window**),
donde cada fold avanza el punto de corte y evalúa las siguientes 53 semanas.

Los folds se ponderan con pesos progresivos `[0.5, 0.75, 1.0, 1.25]`, dando
más importancia a los periodos recientes (2023-2024) y menos al periodo
post-COVID (2020-2021).

La métrica final es `np.average(rmse_folds, weights=cv_weights)` en vez de
`np.mean()`, lo que sesga la selección de hiperparámetros hacia combos que
funcionan bien en datos recientes.

In [ ]:
# --- Fig A3: Diagrama de folds de cross-validation ---
fig, ax = plt.subplots(figsize=(14, 4))

# Simular folds
folds = [
    ('Fold 1', 2014, 2020, 2021, 0.50),
    ('Fold 2', 2014, 2021, 2022, 0.75),
    ('Fold 3', 2014, 2022, 2023, 1.00),
    ('Fold 4', 2014, 2023, 2024, 1.25),
]

for i, (name, train_start, train_end, test_end, weight) in enumerate(folds):
    y = 3 - i
    # Train
    ax.barh(y, train_end - train_start, left=train_start, height=0.5,
            color=IMSS['teal'], alpha=0.7, edgecolor='white')
    # Test
    ax.barh(y, test_end - train_end, left=train_end, height=0.5,
            color=IMSS['burgundy'], alpha=0.7, edgecolor='white')
    # Label
    ax.text(train_start - 0.3, y, f'{name}\n(w={weight})', ha='right', va='center',
            fontsize=9, fontweight='bold')

# COVID zone
ax.axvspan(2020.2, 2022.7, alpha=0.08, color='red')
ax.text(2021.4, 4, 'COVID-19', ha='center', fontsize=8, color='red', alpha=0.6)

# Leyenda
ax.barh([], 0, color=IMSS['teal'], alpha=0.7, label='Entrenamiento')
ax.barh([], 0, color=IMSS['burgundy'], alpha=0.7, label='Evaluación (53 sem)')
ax.legend(loc='upper right')

ax.set_xlabel('Año')
ax.set_yticks([])
ax.set_xlim(2013.5, 2025.5)
ax.set_title('Fig A3. Cross-validation temporal con pesos progresivos',
             fontsize=13, pad=10)
fig.tight_layout()
save_fig(fig, 'figA3_cv_folds')

### FT.4 Protección anti-Newton

Prophet puede caer al optimizador Newton (~100-500x más lento) cuando L-BFGS no converge.
Tres mecanismos lo mitigan:

| Capa | Mecanismo | Efecto |
|:-----|:----------|:-------|
| 1 | Sort CP descendente | Combos con CP alto (rápido) se prueban primero |
| 2 | Timeout por fold (35s) | `ThreadPoolExecutor` corta un fold que exceda 35s |
| 3 | Newton-prone threshold | Si combo con CP=X timeout, skip combos con CP < X |

**Resultado:** Chihuahua-Depresión pasó de 39 min (v4) a 4 min (v5).

### FT.5 Modo híbrido y clasificación de confianza

Series con promedio < 0.5 casos/semana se clasifican como `confianza: insuficiente`.
Con `modelado_hibrido: true` (v6):

1. Se entrenan modelos regionales (4 regiones INEGI de salud mental)
2. Cada estado insuficiente se mapea a su región
3. En predicción, se usa el modelo regional pero se desnormaliza con la **población estatal individual**

**Regiones INEGI de salud mental:**
- Urbana media
- Sur-Sureste vulnerable
- Metropolitana alta
- Rural / dispersa

### FT.6 Periodos atípicos configurados

| Evento | Fecha inicio | Ventana | Efecto |
|:-------|:-------------|:--------|:-------|
| Pandemia COVID-19 | 2020-03-23 | 913 días (~2.5 años) | Holiday global en Prophet |
| Cambio de régimen Tabasco (Depresión) | 2023-01-09 | 365 días | Holiday filtrado por entidad (-6.2% RMSE) |

Los cambios de régimen permanentes (Nayarit, Colima, Durango, BCS) no se modelan
como holidays porque Prophet los trata como eventos temporales, empeorando el RMSE.

### FT.7 Mapa completo de parámetros

| Parámetro | Valor | Fuente |
|:----------|:------|:-------|
| `normalizar_tasa` | `true` | `config/modelado.yaml` |
| `tasa_por` | 100,000 | `config/modelado.yaml` |
| `log_transform` | `true` | `config/modelado.yaml` |
| `TS_SPLITS` | 4 | `config/modelado.yaml` |
| `TEST_SIZE` | 53 semanas | `config/modelado.yaml` |
| `cv_weights` | `[0.5, 0.75, 1.0, 1.25]` | `config/modelado.yaml` |
| `FECHA_CORTE` | 2025-01-01 | `config/modelado.yaml` |
| `umbral_minimo_semanal` | 0.5 | `config/modelado.yaml` |
| `modelado_hibrido` | `true` | `config/params.yaml` |
| `fourier_order` (nacional) | 5 | `config/modelado.yaml` |
| `fourier_order_regional` | 3 | `config/modelado.yaml` |
| `n_changepoints_regional` | 12 | `config/modelado.yaml` |
| `fold_timeout_seconds` | 35 | `config/modelado.yaml` |

</details>

<div style="background: linear-gradient(90deg, #003A70, #006847); height: 3px; margin: 30px 0 15px; border-radius: 2px;"></div>

*Notebook generado como parte del Avance 4 — Modelos Alternativos y Selección del Modelo Final.*

*EpiForecast-MX — Equipo 01 — Febrero 2026*